### ASTR 3300/5300: Astrostatistics
***N. Pol***
___

# Final Exam [65 points total]

### Due: Tuesday, 24 November, 2026, 11:59 pm. NO EXTENSIONS, NO SUBMISSION ACCEPTED PAST DUE DATE/TIME.
---

## Problem 1 [30 points total]

One of the most important revelations in modern cosmology was the accelerated expansion of the Universe, interpreted as being caused by a non-zero Cosmological Constant. In this question you will use a simulated sample of Type Ia supernovae to infer the composition of the Universe. 

1. Load in `astr8070_final_cosmodata.npy`, which can be found in the Github repo under `coursework/final/final_data`. The first column is the SN redshift, the second is the distance modulus, and the third is the Gaussian uncertainty on this distance modulus. Make a labeled scatter plot of the data. [2 points]


2. Plot a line on top of this showing the relationship between the distance modulus and the redshift for the current concordance cosmology. You will need to look up the definition of the distance modulus, look up current reasonable cosmological parameters (cite your source), and plot the relationship. *Hint: check astropy.* [3 points]


3. Create a log-likelihood function that reads in the data and accepts values for the Hubble constant, $H_0$, and the fractional energy density in matter, $\Omega_M$. Assume a flat cosmological geometry, such that $\Omega_M + \Omega_\Lambda=1$. Your function should take $H_0$ and $\Omega_M$, and create a model for the distance moduli values at the redshifts of the samples, then compute the log-likelihood for the data. Print out the log-likelihood value at the cosmological parameter values that you used in (2). For maximum points, make sure your log-likelihood function is vectorized over the data samples. [6 points]


4. Perform an MCMC with appropriate diagnostic checks (either manually yourself, or fine if performed by the sampler you employ) over the $\{H_0, \Omega_M\}$ parameter space. Assume priors of $H_0 \in U[50,100]$ km.s$^{-1}$.Mpc$^{-1}$, and $\Omega_M \in U[0,1]$. Make a labeled corner plot showing the $1$D and $2$D marginalized posterior probability distribution of these parameters. State the median value of the $1$D marginal posteriors for each, along with the values enclosing the $68\%$ credible region. [8 points]


5. Draw $100$ random samples from your posterior chain and plot the corresponding band of solutions that are consistent with the data on the distance-modulus--vs--redshift plot along with the original data. [4 points]


6. Now, using an appropriate technique, find the Bayesian log-evidence of your $\{H_0,\Omega_M\}$ model above. Also find the log-evidence of a model with $\Omega_M=1$ such that $H_0$ is the only varied parameter. What is the Bayes factor for the presence of a Cosmological Constant given these data? [7 points] 

In [1]:
import numpy as np
import corner

from matplotlib import pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format='retina'

from astropy.cosmology import LambdaCDM

In [ ]:
#1.1
#load in the data
data = np.load(r'final_data/astr8070_final_cosmodata.npy')
#label the data
X = data[:, 0]
y = data[:, 1]
yerr = data[:, 2]
#make plot shape
print(data.shape)
#make scatter plot and labels
plt.figure(figsize=(10, 4))
plt.scatter(X,y, color='red')
plt.xlabel('SN Redshift')
plt.ylabel('Distance Modulus')
plt.show()

In [ ]:
#1.2
from astropy.cosmology import LambdaCDM

cosmo = LambdaCDM(H0=67.66, Om0=0.30966, Ode0=0.69034)

xgrid = np.linspace(X.min(), X.max(), 100)
y_fit = cosmo.distmod(xgrid).value

plt.figure(figsize=(10, 4))
plt.scatter(X, y, color='red', label='SN Data')
plt.plot(xgrid, y_fit, color='blue', label='Concordance Cosmology')

plt.xlabel('SN Redshift')
plt.ylabel('Distance Modulus')
plt.legend()
plt.show()

1.2
I used Astropy's LambdaCDM model with H₀ = 67.66 km/s/Mpc and ΩM = 0.30966, based on the Planck 2018 cosmological parameters. Assuming a flat universe, ΩΛ = 1 − ΩM = 0.69034. I then used the model to calculate the predicted distance modulus at different redshifts.

Source: Astropy Collaboration. Planck18 Cosmology. Astropy Documentation. https://docs.astropy.org/en/stable/api/astropy.cosmology.realizations.Planck18.html

In [ ]:
#1.3
def log_likelihood(H0, Om0):
    Ode0 = 1 - Om0

    cosmo = LambdaCDM(H0=H0, Om0=Om0, Ode0=Ode0)
    mu_model = cosmo.distmod(X).value

    logL = -0.5 * np.sum(((y - mu_model) / yerr)**2
                         + np.log(2 * np.pi * yerr**2))

    return logL

print("Log-likelihood =", log_likelihood(67.66, 0.30966))

In [ ]:
#1.4 a)
from PTMCMCSampler import PTMCMCSampler

# Uniform priors
def log_prior(theta):
    H0, Om0 = theta

    if 50 <= H0 <= 100 and 0 <= Om0 <= 1:
        return 0
    return -np.inf

def logL(theta):
    H0, Om0 = theta
    return log_likelihood(H0, Om0)
    
# Starting parameters
theta = np.array([67.66, 0.30966])

# Initial covariance matrix
cov = np.diag([2, 0.05])**2

# Initialize sampler
sampler = PTMCMCSampler.PTSampler(
    2, logL, log_prior, cov,
    outDir='./final_chain'
)

# Run MCMC
sampler.sample(theta, 100000)

# Load chain
chain = np.loadtxt('./final_chain/chain_1.txt')

print("Chain shape =", chain.shape)

In [ ]:
#1.4 b)
# Remove burn-in
burn = int(0.2 * len(chain))
chain_thin = chain[burn:, :2]

# Trace plots
fig, ax = plt.subplots(2, 1, figsize=(10, 5))

ax[0].plot(chain[:, 0])
ax[0].set_ylabel('H0')

ax[1].plot(chain[:, 1])
ax[1].set_ylabel('Omega M')
ax[1].set_xlabel('Sample')

plt.show()


In [ ]:
# Corner plot
fig = corner.corner(
    chain_thin,
    labels=[r'$H_0$', r'$\Omega_M$'],
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True
)

plt.show()

In [ ]:
# Median and 68% credible intervals
H0_values = np.percentile(samples[:, 0], [16, 50, 84])
Om0_values = np.percentile(samples[:, 1], [16, 50, 84])

print("H0 (16%, 50%, 84%) =", H0_values)
print("Omega M (16%, 50%, 84%) =", Om0_values)


In [ ]:
#1.5
# Choose 100 samples from the chain
samples = chain_thin[np.random.choice(len(chain_thin), 100, replace=False)]

xgrid = np.linspace(min(X), max(X), 500)

# Plot the original data
plt.errorbar(X, y, yerr=yerr, fmt='k.')

# Plot 100 burst curves
for theta in samples:
    H0, Om0 = theta
    plt.plot(xgrid, LambdaCDM(H0=H0, Om0=Om0, Ode0=1-Om0).distmod(xgrid).value, color='blue', alpha=0.05)

plt.xlabel('Red Shift')
plt.ylabel('Distance Modulus')
plt.show()

In [ ]:
#1.6
import dynesty

# Model 1: H0 and Omega M
ndim = 2

def loglike(x):
    H0, Om0 = x
    return log_likelihood(H0, Om0)

def ptform(u):
    return np.array([50 + 50*u[0], u[1]])

sampler = dynesty.NestedSampler(loglike, ptform, ndim)
sampler.run_nested()
results = sampler.results

logZ1 = results.logz[-1]
print("Log-evidence with Lambda =", logZ1)


# Model 2: Omega M = 1
ndim = 1

def loglike2(x):
    return log_likelihood(x[0], 1.0)

def ptform2(u):
    return np.array([50 + 50*u[0]])

sampler2 = dynesty.NestedSampler(loglike2, ptform2, ndim)
sampler2.run_nested()
results2 = sampler2.results

logZ2 = results2.logz[-1]
print("Log-evidence without Lambda =", logZ2)


# Bayes factor
logB = logZ1 - logZ2
B = np.exp(logB)

print("Log Bayes factor =", logB)
print("Bayes factor =", B)

1.6
The log-evidence for the model with a cosmological constant in this case lambda, was approximately -269.35, while the model without one was approximately -272.20. So, in conclusion, the Bayes factor came out to be approximately 17.31, which means the data favors a model with a cosmological constant.

## Problem 2 [35 points total]

There are several empirical scaling relationships between the masses of black holes in galaxy centers and the properties of their hosts. In this problem you will use real data from the "MASSIVE" Galaxy Survey. This data updates that used in [McConnell & Ma (2013)](https://ui.adsabs.harvard.edu/abs/2013ApJ...764..184M/abstract), which presented a systematic investigation of scaling relations with different catalog subsets.

1. Download this dataset: http://blackhole.berkeley.edu/wp-content/uploads/2016/09/current_ascii.txt. Skip the appropriate header rows and read this into a `pandas` dataframe. Use the meta-data in the header rows to create appropriate column names for your dataframe. Make sure that your data file is properly formatted with tab-spaced columns so that it is readable without errors. [3 points]


2. Create a data matrix out of the base-10 log values of: the galaxy distance, the velocity dispersion $\sigma$, and the black hole mass. Create a target vector that has $1$ for early-type galaxies (morphology has `E` or `S0` in the value) and $0$ for late-type galaxies. Print out the number of early-type and late-type galaxies that are in your target vector, as well as the total number of galaxies. [3 points]


3. Perform a 50-50 train-test split on the data matrix and target vector, using `random_state=0` for reproducibility. Train the following classifiers, overplotting ROC curves for all:
    - Gaussian Naive Bayes
    - LDA
    - QDA
    - $K$-nearest neighbors (with $K=10$)
    - Gaussian mixture model (with $2$ components)
    - Decision Tree (with `entropy` criterion, and tree depth $=2$)
 [5 points]
    

4. What minimum tree depth do you need to achieve essentially perfect classification accuracy with the Decision Tree? Confirm this by making a two-panel figure with scatter plots of the log10-distance and log10-sigma from the test data, color-coded by the test label (left panel) and predicted label (right panel).  [3 points]


5. The following regression tasks should all produce best-fit regression coefficients and a plot. State how many galaxies are being fit in each scenario.
    - Use the $68\%$ low and high black-hole mass values to deduce the uncertainty on each $(M_\mathrm{BH}/M_\odot)$. Convert this into an uncertainty on $\log_{10}(M_\mathrm{BH}/M_\odot)$. Ignoring the uncertainties on $\sigma$, perform linear regression of the form
$$ \log_{10}(M_\mathrm{BH}/M_\odot) = \alpha + \beta\log_{10}(\sigma \,/\, 200\,\mathrm{km.s}^{-1})$$
and state the best-fit regression coefficients $\{\alpha,\beta\}$. Plot the best-fit line on a scatter plot with the data. 
    - Repeat the previous bullet to fit and plot the relation $$ \log_{10}(M_\mathrm{BH}/M_\odot) = \alpha + \beta\log_{10}(L_V \,/\, 10^{11}L_\odot)$$ for the sub-sample of galaxies with $V$-band luminosity data (ignoring the $L_V$ uncertainties).
    - Repeat to fit and plot the relation $$ \log_{10}(M_\mathrm{BH}/M_\odot) = \alpha + \beta\log_{10}(M_\mathrm{bulge} \,/\, 10^{11}M_\odot)$$ for the sub-sample of galaxies with bulge mass measurements (ignoring the $M_\mathrm{bulge}$ uncertainties).
 [7 points]
 
 
6. Perform polynomial regression of the form
$$ \log_{10}(M_\mathrm{BH}/M_\odot) = \alpha + \sum_{p=1}^{N_p}\beta_p[\log_{10}(\sigma \,/\, 200\,\mathrm{km.s}^{-1})]^p.$$
For $N_p$ in the range $0$ to $5$, fit the polynomial relationship, print the regression coefficients, and compute the BIC value. Make a labeled plot of the BIC versus the polynomial degree. Is a quadratic or higher polynomial relationship between $\log_{10}(M_\mathrm{BH}/M_\odot)$ and $\log_{10}(\sigma \,/\, 200\,\mathrm{km.s}^{-1})$ favored? [5 points]


7. Perform linear regression ($N_p=1$) on the dataset on sub-samples of early-type and late-type galaxies. Print the best-fit regression coefficients from each of these sub-sets. Make a scatter plot that includes the original data color coded by early/late type, the fit to all of the data, and the fits from the early/late-type sub-samples. The plot should have appropriate labels and a legend. [4 points]

In [ ]:
#2.1
import pandas as pd

columns = [
    'Galaxy', 'Distance', 'MBH', 'MBH_lower', 'MBH_upper',
    'method', 'sigma', 'sigma_lower', 'sigma_upper',
    'logLV', 'logLV_err', 'logL36', 'logL36_err',
    'Mbulge', 'radius', 'Morphology', 'Profile',
    'Reff_V', 'Reff_i', 'Reff_36'
]

data = pd.read_csv(
    'final_data/astr8070_final_mcma.txt',
    sep=r'\s+',
    skiprows=23,
    names=columns)

print(data.head())
print(data.shape)

In [ ]:
#2.2
# Galaxy types
early = data['Morphology'].str.startswith(('E', 'S0'))
late = data['Morphology'].str.startswith('S') & ~data['Morphology'].str.startswith('S0')

# Select valid data
valid = (data['Distance'] > 0) & (data['sigma'] > 0) & (data['MBH'] > 0)
galaxies = data[valid & (early | late)]

# Data matrix
X = np.vstack([np.log10(galaxies['Distance']),
               np.log10(galaxies['sigma']),
               np.log10(galaxies['MBH'])]).T

# Target vector
y = np.zeros(len(galaxies), dtype=int)
y[galaxies['Morphology'].str.startswith(('E', 'S0')).to_numpy()] = 1

print("Early-type =", np.sum(y == 1))
print("Late-type =", np.sum(y == 0))

In [ ]:
#2.3
#50-50 train-test-split on data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)
from astroML.classification import GMMBayes
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc

from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from astroML.classification import GMMBayes

names = ['GNB', 'LDA', 'QDA', 'KNN', 'GMM Bayes', 'Decision Tree']

models = [GaussianNB(),
          LinearDiscriminantAnalysis(),
          QuadraticDiscriminantAnalysis(),
          KNeighborsClassifier(n_neighbors=10),
          GMMBayes(n_components=2),
          DecisionTreeClassifier(criterion='entropy', max_depth=2, random_state=0)]

scores = []

for i in range(len(models)):
    clf = models[i]
    clf.fit(X_train, y_train)

    y_prob = clf.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)

    scores.append(auc(fpr, tpr))
    plt.plot(fpr, tpr, label=names[i])

plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC Curves for Galaxy Classification')
plt.legend()
plt.show()

print("Best classifier =", names[np.argmax(scores)])
print("Best AUC =", max(scores))

In [ ]:
#2.4
from sklearn.metrics import accuracy_score

clf = DecisionTreeClassifier(criterion='entropy', max_depth=2, random_state=0)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Decision Tree Accuracy =", accuracy_score(y_test, y_pred))

fig = plt.figure(figsize=(10, 4))

ax = fig.add_subplot(121)
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test)
ax.set_xlabel('log10(Distance)')
ax.set_ylabel('log10(sigma)')
ax.set_title('True Labels')

ax = fig.add_subplot(122)
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_pred)
ax.set_xlabel('log10(Distance)')
ax.set_ylabel('log10(sigma)')
ax.set_title('Predicted Labels')

plt.show()

In [ ]:
#2.5a
from astroML.linear_model import LinearRegression

galaxies = data[(data['MBH'] > 0) &
                (data['MBH_lower'] > 0) &
                (data['MBH_upper'] > 0) &
                (data['sigma'] > 0)]

x = np.log10(galaxies['sigma'].to_numpy() / 200)
y = np.log10(galaxies['MBH'].to_numpy())

# Uncertainty in log10(MBH)
yerr = (np.log10(galaxies['MBH_upper'].to_numpy()) -
        np.log10(galaxies['MBH_lower'].to_numpy())) / 2

# Linear regression
linreg = LinearRegression()
linreg.fit(x[:, None], y, yerr)

print("Best-fit coefficients =", linreg.coef_)
print("alpha =", linreg.coef_[0])
print("beta =", linreg.coef_[1])

# Best-fit line
xgrid = np.linspace(min(x), max(x), 100)
y_fit = linreg.predict(xgrid[:, None])

plt.figure(figsize=(8, 5))
plt.errorbar(x, y, yerr=yerr, fmt='k.', label='Galaxy Data')
plt.plot(xgrid, y_fit, 'r-', label='Linear Fit')

plt.xlabel(r'$\log_{10}(\sigma/200)$')
plt.ylabel(r'$\log_{10}(M_{BH}/M_{\odot})$')
plt.title('Black Hole Mass vs. Velocity Dispersion')
plt.legend()
plt.show()

In [ ]:
#2.5b

galaxies = data[(data['MBH'] > 0) &
                (data['logLV'] > 0)]

x = galaxies['logLV'].to_numpy() - 11
y = np.log10(galaxies['MBH'].to_numpy())

# Linear regression
linreg = LinearRegression()
linreg.fit(x[:, None], y)

print("Best-fit coefficients =", linreg.coef_)
print("alpha =", linreg.coef_[0])
print("beta =", linreg.coef_[1])

# Best-fit line
xgrid = np.linspace(min(x), max(x), 100)
y_fit = linreg.predict(xgrid[:, None])

plt.figure(figsize=(8, 5))
plt.scatter(x, y, color='red', label='Galaxy Data')
plt.plot(xgrid, y_fit, 'b-', label='Linear Fit')

plt.xlabel(r'$\log_{10}(L_V/10^{11}L_{\odot})$')
plt.ylabel(r'$\log_{10}(M_{BH}/M_{\odot})$')
plt.title('Black Hole Mass vs. V-band Luminosity')
plt.legend()
plt.show()

In [ ]:
#2.5c

galaxies = data[(data['MBH'] > 0) &
                (data['Mbulge'] > 0)]

x = np.log10(galaxies['Mbulge'].to_numpy() / 1e11)
y = np.log10(galaxies['MBH'].to_numpy())

# Linear regression
linreg = LinearRegression()
linreg.fit(x[:, None], y)

print("Best-fit coefficients =", linreg.coef_)
print("alpha =", linreg.coef_[0])
print("beta =", linreg.coef_[1])

# Best-fit line
xgrid = np.linspace(min(x), max(x), 100)
y_fit = linreg.predict(xgrid[:, None])

plt.figure(figsize=(8, 5))
plt.scatter(x, y, color='red', label='Galaxy Data')
plt.plot(xgrid, y_fit, 'b-', label='Linear Fit')

plt.xlabel(r'$\log_{10}(M_{bulge}/10^{11}M_{\odot})$')
plt.ylabel(r'$\log_{10}(M_{BH}/M_{\odot})$')
plt.title('Black Hole Mass vs. Bulge Mass')
plt.legend()
plt.show()

In [ ]:
#2.6
from astroML.linear_model import PolynomialRegression

# Same data as 2.5a
galaxies = data[(data['MBH'] > 0) &
                (data['MBH_lower'] > 0) &
                (data['MBH_upper'] > 0) &
                (data['sigma'] > 0)]

x = np.log10(galaxies['sigma'].to_numpy() / 200)
y = np.log10(galaxies['MBH'].to_numpy())

yerr = (np.log10(galaxies['MBH_upper'].to_numpy()) -
        np.log10(galaxies['MBH_lower'].to_numpy())) / 2

# Polynomial regression for degrees 0 through 5
degrees = np.arange(0, 6)
BIC = []

for i in degrees:
    poly = PolynomialRegression(degree=i)
    poly.fit(x[:, None], y, yerr)

    y_fit = poly.predict(x[:, None])

    chi2 = np.sum(((y - y_fit) / yerr)**2)
    bic = chi2 + (i + 1)*np.log(len(y))

    BIC.append(bic)

    print("Degree =", i, "Coefficients =", poly.coef_, "BIC =", bic)

# Plot BIC
plt.figure(figsize=(8, 5))
plt.plot(degrees, BIC, 'o-')

plt.xlabel('Polynomial Degree')
plt.ylabel('BIC')
plt.title('BIC vs. Polynomial Degree')
plt.show()

print("Best polynomial degree =", degrees[np.argmin(BIC)])

2.6
From the BIC values, the fifth-degree polynomial had the lowest value at about 17,590.73. This means that a higher-order polynomial is favored over the linear model for this data.

In [ ]:
#2.7
from astroML.linear_model import LinearRegression

# Same data as 2.5a
galaxies = data[(data['MBH'] > 0) &
                (data['MBH_lower'] > 0) &
                (data['MBH_upper'] > 0) &
                (data['sigma'] > 0)]

# Identify galaxy types
early = galaxies['Morphology'].str.startswith(('E', 'S0')).to_numpy()
late = (galaxies['Morphology'].str.startswith('S') &
        ~galaxies['Morphology'].str.startswith('S0')).to_numpy()

# Variables
x = np.log10(galaxies['sigma'].to_numpy() / 200)
y = np.log10(galaxies['MBH'].to_numpy())

yerr = (np.log10(galaxies['MBH_upper'].to_numpy()) -
        np.log10(galaxies['MBH_lower'].to_numpy())) / 2

# Linear regression for all galaxies
linreg_all = LinearRegression()
linreg_all.fit(x[:, None], y, yerr)

# Linear regression for early-type galaxies
linreg_early = LinearRegression()
linreg_early.fit(x[early, None], y[early], yerr[early])

# Linear regression for late-type galaxies
linreg_late = LinearRegression()
linreg_late.fit(x[late, None], y[late], yerr[late])

# Print coefficients
print("All galaxies =", linreg_all.coef_)
print("Early-type galaxies =", linreg_early.coef_)
print("Late-type galaxies =", linreg_late.coef_)

# Predictions
xgrid = np.linspace(min(x), max(x), 100)

y_fit_all = linreg_all.predict(xgrid[:, None])
y_fit_early = linreg_early.predict(xgrid[:, None])
y_fit_late = linreg_late.predict(xgrid[:, None])

# Plot
plt.figure(figsize=(10, 5))

plt.scatter(x[early], y[early], color='blue', label='Early-type Galaxies')
plt.scatter(x[late], y[late], color='red', label='Late-type Galaxies')

plt.plot(xgrid, y_fit_all, color='black', label='All Galaxies Fit')
plt.plot(xgrid, y_fit_early, 'b--', label='Early-type Fit')
plt.plot(xgrid, y_fit_late, 'r--', label='Late-type Fit')

plt.xlabel(r'$\log_{10}(\sigma/200)$')
plt.ylabel(r'$\log_{10}(M_{BH}/M_{\odot})$')
plt.title('Black Hole Mass vs. Velocity Dispersion')
plt.legend()
plt.show()